<a href="https://colab.research.google.com/github/DiasFrazerGroup/Intro-to-probabilistic-modelling-2026/blob/main/Session2a_HierarchicalModels_WithAnswers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hierarchical Models: A Clinical Trial Meta-Analysis
*Session 2a — An Introduction to Probabilistic Model Building*

---

## The real-world problem that motivated this

In the 1970s–1980s, many clinical trials tested beta-blockers after heart attacks.  
Each trial was small and showed mixed results — some positive, some null, some even negative.  
Doctors were confused: does the drug work or not?

A hierarchical Bayesian meta-analysis resolved the contradiction:  
the drug *does* work on average, but the effect genuinely varies across patient populations and trial settings.  
Small trials were just too noisy to detect the signal reliably.

**Meta-analysis** — combining results across multiple studies — is now standard practice in evidence-based medicine.  
The hierarchical model is the right statistical tool for it.

---

## Our scenario

Twelve clinical trials have tested a new blood-pressure drug. Each trial reports an **estimated effect size** — the mean reduction in systolic blood pressure (mmHg) — along with the number of patients enrolled.

Trials vary enormously in size: some enrolled 12 patients, some enrolled 500.  
The key question: **how should we estimate the effect in each trial, and the overall drug effect?**

| Strategy | Model | Problem |
|---|---|---|
| **No pooling** | Estimate each trial independently | Small trials are wildly uncertain |
| **Complete pooling** | One shared effect for all trials | Ignores real between-trial variation |
| **Partial pooling** | Hierarchical: trial effects drawn from a population | Borrows strength across trials |

---

## Where this fits in the course

| Direction | Session | Concept | How it connects |
|---|---|---|---|
| ← Session 1 | Foundations | Prior / posterior | Population prior on $\theta_k$; each trial's data updates it |
| ← Session 1 | Foundations | Latent variable model | Each trial's true effect $\theta_k$ is a latent variable — never directly observed |
| ← Session 1 | Foundations | DAG notation | `μ, τ → θ_k → y_k` — same DAG language from the lecture |
| → Session 2b | Protein VAE | Same model, learned likelihood | VAE = same latent variable DAG; decoder is a neural net instead of Normal |
| → Session 3b | Compressed sensing | Bayesian inference on a sparse latent | Same posterior machinery; the unknown is a sparse activity vector |

**Session 2b connection in detail:** the hierarchical model and the VAE share the exact same structure:
latent variable $z$ per data point, population prior $p(z)$, and a likelihood $p(\mathbf{x}|z)$.
In the hierarchical model $p(\mathbf{x}|z)$ is Normal with known $\sigma$.
In the VAE it is a deep neural network — same idea, learned likelihood.

In [ ]:
!pip install pymc arviz matplotlib numpy --quiet

In [ ]:
import logging
logging.getLogger('pymc').setLevel(logging.ERROR)
logging.getLogger('pytensor').setLevel(logging.ERROR)

import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('PyMC:', pm.__version__)
print('ArviZ:', az.__version__)

## 1. The data

In [ ]:
import numpy as np

# Reproducibility: fix the random seed so results are the same every run.
rng = np.random.default_rng(42)

K = 12   # number of trials — small enough to visualise clearly

# ── True population parameters ────────────────────────────────────────────────
# In real life these are unknown; we generate them here so we can check answers.
true_mu  = 10.0   # overall drug effect: 10 mmHg mean reduction in systolic BP
true_tau =  4.0   # between-trial SD: trials differ from each other by ~4 mmHg
                  # (reflects different patient populations, doses, comorbidities)

# ── Per-trial true effects: the latent variables θ_k ─────────────────────────
# Each trial's true effect is drawn from the population distribution N(μ, τ).
# These are NEVER observed — the whole point of the model is to infer them.
true_theta_k = rng.normal(true_mu, true_tau, size=K)

# ── Trial sizes — spanning two orders of magnitude on purpose ─────────────────
# Real meta-analyses include a mix of pilot studies (n~10) and large RCTs (n~500).
n_patients = np.array([15, 400, 25, 300, 12, 500, 50, 80, 120, 20, 200, 30])

# ── Known standard errors: SE_k = σ_patient / √n_k ───────────────────────────
# σ_patient = typical within-patient variability of systolic BP (~20 mmHg).
# The standard error of the trial mean shrinks as √n — larger trials are more precise.
# Crucially, σ_k is KNOWN from the trial's sample size (no need to estimate it).
sigma_patient = 20.0
sigma_k = sigma_patient / np.sqrt(n_patients)

# ── Observed trial estimates ──────────────────────────────────────────────────
# Each trial reports one summary number: mean BP reduction across its patients.
# That number equals the true effect plus Gaussian noise with known SD = σ_k.
y_obs = rng.normal(true_theta_k, sigma_k)

# ── Print the data table ──────────────────────────────────────────────────────
print(f'  {"Trial":6s}  {"n":>5}  {"SE":>6}  {"Observed effect":>16}  {"True effect":>12}')
print('  ' + '-' * 55)
for k in range(K):
    print(f'  T{k+1:<5d}  {n_patients[k]:5d}  {sigma_k[k]:6.2f}  '
          f'{y_obs[k]:+8.2f} mmHg        {true_theta_k[k]:+7.2f} mmHg')
print()
print(f'True overall effect (μ): {true_mu:.1f} mmHg  |  True heterogeneity (τ): {true_tau:.1f} mmHg')
print()
print('Notice: small trials (n=12, 15) have large SEs — their observed effects')
print('can be far from the truth. Partial pooling will correct for this.')



### Generative model

```
μ   ~ Normal(0, 20)      ← overall drug effect (broad prior)
τ   ~ HalfNormal(10)     ← between-trial heterogeneity
                           (how much trials genuinely differ from each other)

θ_k ~ Normal(μ, τ)       ← true effect in trial k  (latent — never directly observed)

y_k ~ Normal(θ_k, σ_k)   ← observed effect from trial k
                            σ_k = σ / √n_k  is KNOWN from the trial's sample size
```

The fact that $\sigma_k$ is known (rather than estimated) is standard in meta-analysis: each trial reports its own standard error, derived from the number of patients and the variance of blood pressure measurements.

### What should we expect from the models?

A trial with n=12 patients and within-patient variability σ=20 mmHg has a standard error of 20/√12 ≈ 5.8 mmHg — more than half the true drug effect. Its observed result is dominated by noise. The no-pooling model will accept that noisy result at face value. The hierarchical model will recognise the uncertainty and pull its estimate toward what the other trials suggest. This is called **shrinkage** — and we'll see exactly how it works after fitting both models.

In [ ]:
with pm.Model() as hier_model:
    mu      = pm.Normal('mu',      mu=0, sigma=20)
    tau     = pm.HalfNormal('tau', sigma=10)
    theta_k = pm.Normal('theta_k', mu=mu, sigma=tau, shape=12)
    se_k    = pm.Data('se_k', sigma_k)          # now visible in the DAG
    y       = pm.Normal('y', mu=theta_k, sigma=se_k, observed=y_obs)

pm.model_to_graphviz(hier_model)

## 2. No-pooling model

Each trial gets a completely independent estimate with a vague prior — no information shared.

```
θ_k ~ Normal(0, 100)    ← independent vague prior for each trial
y_k ~ Normal(θ_k, σ_k)  ← σ_k fixed (known from trial size)
```

Because σ_k is known and the prior is vague, the posterior for each trial is approximately:  
$\theta_k | y_k \approx \mathcal{N}(y_k,\; \sigma_k^2)$

In other words: **the no-pooling posterior mean = observed effect, posterior SD = reported SE**.  
Small trials will have wide, uncertain posteriors — their one or two observations could be anywhere.

In [ ]:
# The no-pooling model treats each trial as a completely separate study.
# There is no communication between trials — each θ_k is estimated from
# that trial's data alone, ignoring all the other 11 trials.

with pm.Model() as no_pool_model:

    # Independent vague prior for each trial's effect.
    # N(0, 100) is very broad — it says almost any effect from -300 to +300 mmHg
    # is plausible a priori. In practice, the posterior will be dominated by the data.
    theta_k = pm.Normal('theta_k', mu=0, sigma=100, shape=K)

    # Likelihood: the observed trial mean is normally distributed around
    # the true effect, with known SE = σ_k (fixed, not estimated).
    y = pm.Normal('y', mu=theta_k, sigma=sigma_k, observed=y_obs)

    # MCMC: draw 1000 samples after 1000 warm-up (tuning) steps.
    # chains=2 lets us check that two independent chains agree (convergence check).
    idata_no_pool = pm.sample(
        draws=1000, tune=1000, chains=2, random_seed=42,
        progressbar=True, target_accept=0.9
    )

# Posterior summary: mean and SD across all MCMC samples.
# Because the prior is vague, these are essentially:
#   posterior mean ≈ y_obs[k]   (the raw observation)
#   posterior SD   ≈ sigma_k[k] (the known SE)
mu_no_pool = idata_no_pool.posterior['theta_k'].mean(dim=('chain','draw')).values
sd_no_pool = idata_no_pool.posterior['theta_k'].std(dim=('chain','draw')).values

print('No-pooling posterior means (should match observed effects closely):')
for k in range(K):
    print(f'  T{k+1}: {mu_no_pool[k]:+.2f} ± {sd_no_pool[k]:.2f}   (observed: {y_obs[k]:+.2f})')


## 3. Hierarchical (partial pooling) model

Now we add the population level. Each trial's true effect is drawn from a shared Normal distribution,  
and we infer both the population mean $\mu$ and the between-trial heterogeneity $\tau$ from the data.

```
μ   ~ Normal(0, 20)     ← overall drug effect
τ   ~ HalfNormal(10)    ← between-trial heterogeneity

θ_k ~ Normal(μ, τ)      ← trial k's true effect — drawn from the population
y_k ~ Normal(θ_k, σ_k)  ← observed effect (σ_k fixed)
```

**What changes:** small trials, which have large $\sigma_k$, will have their $\theta_k$ pulled toward $\mu$.  
Large trials, which have precise estimates, will barely move.  
The model *automatically* figures out how much to trust each trial based on its size.

In [ ]:
# The hierarchical model adds a "population level" above the individual trials.
# Instead of independent priors, each trial's true effect θ_k is drawn from
# a shared population distribution N(μ, τ). The model learns μ and τ from data,
# then uses that knowledge to sharpen estimates for each individual trial.

with pm.Model() as hier_model:

    # Population-level parameters — inferred jointly from all 12 trials.
    mu  = pm.Normal('mu',  mu=0, sigma=20)    # overall drug effect (broad prior)
    tau = pm.HalfNormal('tau', sigma=10)       # between-trial SD (must be ≥ 0)
                                               # HalfNormal concentrates mass near 0
                                               # but allows large values if data demands

    # Trial-level effects — each drawn from the population distribution.
    # This is what "hierarchical" means: θ_k is constrained to be near μ,
    # but allowed to deviate by τ. The more uncertain trial k is (large σ_k),
    # the more its estimate will be pulled toward μ.
    theta_k = pm.Normal('theta_k', mu=mu, sigma=tau, shape=K)

    # Likelihood: same as no-pooling — observed effect is true effect + noise.
    y = pm.Normal('y', mu=theta_k, sigma=sigma_k, observed=y_obs)

    idata_hier = pm.sample(
        draws=1000, tune=1000, chains=2, random_seed=42,
        progressbar=True, target_accept=0.9
    )

# Extract posterior means for all parameters.
# mu_hier[k] is the hierarchical estimate of trial k's true effect.
mu_hier  = idata_hier.posterior['theta_k'].mean(dim=('chain','draw')).values
sd_hier  = idata_hier.posterior['theta_k'].std(dim=('chain','draw')).values

# Population-level estimates: overall effect and between-trial heterogeneity.
mu_pop   = float(idata_hier.posterior['mu'].mean())
tau_est  = float(idata_hier.posterior['tau'].mean())

print(f'Estimated overall effect μ: {mu_pop:.2f} mmHg  (true: {true_mu:.1f})')
print(f'Estimated heterogeneity  τ: {tau_est:.2f} mmHg  (true: {true_tau:.1f})')


## 3b. Baseline comparison: does pooling actually help?

Before explaining *why* the hierarchical model works, we should check *that* it works —  
and compare both models against the simplest possible alternative.

**Three approaches, ordered by how much information they share across trials:**

| Approach | Per-trial estimate | Pooling |
|---|---|---|
| **Complete pooling** | Same value for every trial (weighted average) | Total |
| **Partial pooling** | Hierarchical posterior mean — between no-pool and mean | Partial |
| **No pooling** | Raw observation y_k — no information shared | None |

The **precision-weighted average** (standard fixed-effects meta-analysis) weights each trial  
by 1/σ_k², so large precise trials dominate. It's the right estimator for the global mean *if*  
all trials measure exactly the same true effect (τ = 0).

Applying that single number to every trial gives the **complete-pooling estimate**.  
It's the simplest baseline: can the hierarchical model do better than this?

In [ ]:
# ── Precision-weighted (fixed-effects) global mean ───────────────────────────
# Each trial is weighted by 1/σ_k² = its precision (more precise = more weight).
# This is the standard fixed-effects meta-analysis estimator — valid if τ = 0.
weights   = 1.0 / sigma_k**2
mu_fixed  = np.average(y_obs, weights=weights)  # precision-weighted mean

# Unweighted mean for comparison — gives equal weight to a 12-patient and a 500-patient trial.
mu_simple = np.mean(y_obs)

# Complete pooling: apply the single global estimate to every trial.
# This ignores all between-trial variation — if τ > 0, it's wrong for every trial.
complete_pool = np.full(K, mu_fixed)

# ── Per-trial errors for all three approaches ─────────────────────────────────
err_complete = np.abs(complete_pool - true_theta_k)  # complete pooling
err_nop      = np.abs(mu_no_pool    - true_theta_k)  # no pooling (MCMC)
err_hier_obs = np.abs(mu_hier       - true_theta_k)  # partial pooling (MCMC)

# ── Print summary ─────────────────────────────────────────────────────────────
print("Estimating the GLOBAL drug effect (μ):")
print(f"  Simple unweighted average:  {mu_simple:.2f} mmHg")
print(f"  Precision-weighted average: {mu_fixed:.2f} mmHg  ← up-weights large trials")
print(f"  Hierarchical posterior μ:   {mu_pop:.2f} mmHg")
print(f"  True value:                 {true_mu:.1f} mmHg")
print()
print("Per-trial mean absolute error (MAE = |estimate − true effect|):")
print(f"  Complete pooling (fixed effects): {err_complete.mean():.2f} mmHg")
print(f"  No pooling:                       {err_nop.mean():.2f} mmHg")
print(f"  Partial pooling (hierarchical):   {err_hier_obs.mean():.2f} mmHg")
print()

# Break down by trial size — the benefit should be concentrated in small trials.
for label, mask in [("Small trials (n ≤ 50)",  n_patients <= 50),
                    ("Large trials (n > 50)",   n_patients > 50)]:
    print(f"{label}:")
    print(f"  Complete pooling: {err_complete[mask].mean():.2f} mmHg")
    print(f"  No pooling:       {err_nop[mask].mean():.2f} mmHg")
    print(f"  Hierarchical:     {err_hier_obs[mask].mean():.2f} mmHg")
    print()

print("Note: this is ONE random draw. Results will vary across runs.")
print("The simulation in Section 5 shows the expected improvement across many datasets.")


## 4. Why does the hierarchical model do better? Shrinkage explained.

### The intuition: a thought experiment

Imagine trial T5 (n=12, σ_k ≈ 5.8 mmHg) observed an effect of +22 mmHg. What should you believe?

There are two explanations:
1. This population genuinely responds more strongly to the drug — true effect really is ~22 mmHg
2. It's a noisy measurement — with SE ≈ 5.8 mmHg, a lucky draw could easily produce +22 even if the true effect is only 10 mmHg

The hierarchical model weighs these two explanations using the evidence from all the other trials. Because the other trials cluster around 10 mmHg, explanation (2) is more plausible for a small trial. The model's estimate for trial T5 is therefore pulled partway back toward the population mean — this pull is called **shrinkage**.

### The formula

When the likelihood and prior are both Normal (as they are here), the posterior mean has a closed form:

$$\hat{\theta}_k = \underbrace{\mu}_{\text{population mean}} + \underbrace{\frac{\tau^2}{\tau^2 + \sigma_k^2}}_{\text{reliability}} \times (y_k - \mu)$$

- **Reliability** = how much to trust this trial's own data. High when the trial is precise (σ_k ≪ τ); low when the trial is noisy (σ_k ≫ τ).
- When reliability = 1: $\hat{\theta}_k = y_k$ — large trial, use its own data entirely
- When reliability = 0: $\hat{\theta}_k = \mu$ — tiny trial, fall back on the population mean
- In between: a precision-weighted average of both sources of information

The **shrinkage fraction** — how far the estimate is pulled toward μ — is $1 - \text{reliability}$:

$$B_k = \frac{\sigma_k^2}{\sigma_k^2 + \tau^2}$$

### Is this a big correction or a small one?

For a **small trial (n = 12)**: σ_k ≈ 5.8 mmHg, τ ≈ 4 mmHg.
- Reliability = 4²/(4² + 5.8²) = 16/49.6 ≈ **32%**
- Shrinkage fraction = **68%** — the estimate is pulled more than halfway toward the population mean
- Expected squared error — no-pooling: σ_k² ≈ 33.6 mmHg².  Hierarchical: σ_k² × reliability ≈ **10.8 mmHg². That's a 3× reduction in expected error.**

For a **large trial (n = 500)**: σ_k ≈ 0.9 mmHg, τ ≈ 4 mmHg.
- Reliability ≈ 95%, shrinkage fraction ≈ 5% — barely touched
- Expected squared error: 0.8 → 0.76 mmHg². A 5% improvement — negligible (these trials are already precise).

**The correction is large exactly where it matters: small, noisy trials.**

### Does the hierarchical model only help the global mean estimate, or also per-trial estimates?

**Both — but they work differently.**

**Per-trial estimates (the individual θ_k):** The shrinkage formula shows the hierarchical estimator has *lower expected squared error* than the raw observation for **every** trial. For small trials the improvement is large (3× for n=12); for large trials it's negligible. The mechanism — pulling toward the population mean — is precisely the improvement. A noisy trial benefits from "borrowing" information from the other trials.

**Global mean estimate (μ):** The hierarchical model estimates the overall drug effect as a reliability-weighted combination of all trials. This correctly down-weights small noisy trials, producing a better estimate of μ than a naive average that treats n=12 and n=500 equally.

### One important caveat

In any **single run**, shrinkage may hurt for some individual trials — specifically those whose true effect is genuinely far from the population mean. A trial that truly has an unusual effect will get pulled toward the centre, worsening its estimate for that run. Whether shrinkage helps or hurts for a specific trial depends on whether that trial's true θ_k happens to be close to μ. The statistical guarantee is about **expected error over many realisations** — see the simulation at the end of Section 5.

In [ ]:
# ── Analytic posterior mean formula ───────────────────────────────────────────
# For Normal-Normal conjugacy, the posterior mean is a closed-form expression.
# We use the MCMC estimates of mu and tau (computed above) as the population parameters.

# Reliability: weight on the trial's own data. Approaches 1 as σ_k → 0 (large trial).
reliability = tau_est**2 / (tau_est**2 + sigma_k**2)

# Shrinkage fraction: weight on the population mean. Approaches 1 as σ_k → ∞ (tiny trial).
shrinkage_frac = 1.0 - reliability

# Analytic posterior mean: weighted average of observed effect and population mean.
# This should closely match the MCMC posterior means (mu_hier).
analytic_shrunk = mu_pop + reliability * (y_obs - mu_pop)

# Expected MSE from the analytic formula (averaging over the observation noise):
#   E[(θ̂_k − θ_k)²] = σ_k² × τ² / (σ_k² + τ²) = σ_k² × reliability
# For no-pooling (θ̂_k = y_k):
#   E[(y_k − θ_k)²] = σ_k²
# So the ratio is just the reliability — hierarchical is always better in expectation.
expected_mse_nop  = sigma_k**2
expected_mse_hier = sigma_k**2 * reliability   # = σ_k² × τ² / (σ_k²+τ²)
mse_ratio         = expected_mse_hier / expected_mse_nop   # always < 1

print(f"Using fitted population parameters: μ = {mu_pop:.2f} mmHg, τ = {tau_est:.2f} mmHg\n")
print(f"  {'Trial':6s}  {'n':>5}  {'SE':>5}  {'Shrinkage':>10}  {'Expected MSE ratio':>20}  "
      f"{'Observed':>9}  {'Analytic':>9}  {'PyMC':>7}  {'True':>7}")
print("  " + "-" * 95)
for k in range(K):
    print(f"  T{k+1:<5d}  {n_patients[k]:5d}  {sigma_k[k]:5.2f}  "
          f"{shrinkage_frac[k]*100:7.0f}%     "
          f"  {mse_ratio[k]:.2f} (hier/no-pool)   "
          f"{y_obs[k]:+9.2f}  {analytic_shrunk[k]:+9.2f}  "
          f"{mu_hier[k]:+7.2f}  {true_theta_k[k]:+7.2f}")

print()
print("MSE ratio < 1 means hierarchical has lower EXPECTED squared error.")
print("For n=12: ratio ≈ 0.32 → hierarchical expected error is 3× lower than no-pooling.")
print()
print("Analytic formula vs PyMC posterior means: should be very close.")


## 5. Visualising shrinkage and running a simulation

**Two plots:**
- **Left — forest plot**: estimates from all three approaches side by side. Arrows of shrinkage are visible for small trials.
- **Right — simulation**: repeat the whole experiment 300 times with fresh random data (same trial sizes, same population parameters). Average the error at each trial size. This removes the noise from a single run and shows the **expected** improvement.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── 95% HDI (highest density interval) for each trial ─────────────────────────
# arviz.hdi computes a Bayesian credible interval from the MCMC samples.
# This is narrower for large trials (less uncertainty) and wider for small ones.
hdi_no_pool = az.hdi(idata_no_pool, var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values  # (K, 2)
hdi_hier    = az.hdi(idata_hier,    var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values

# Sort trials by size (largest at top, as is convention in forest plots).
order = np.argsort(n_patients)
y_pos = np.arange(K)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left panel: forest plot ────────────────────────────────────────────────────
ax = axes[0]
for i, k in enumerate(order):
    # No-pooling CI (orange) — wide for small trials, narrow for large ones.
    ax.plot([hdi_no_pool[k,0], hdi_no_pool[k,1]], [y_pos[i]+0.25]*2,
            color='C1', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_no_pool[k], y_pos[i]+0.25, s=55, color='C1', zorder=5)

    # Hierarchical CI (blue) — should be narrower for small trials (borrowing strength).
    ax.plot([hdi_hier[k,0], hdi_hier[k,1]], [y_pos[i]-0.25]*2,
            color='C0', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_hier[k], y_pos[i]-0.25, s=55, color='C0', zorder=5)

    # Complete-pooling estimate: a single vertical dotted line per trial.
    # This is identical for every trial — it shows how much info is lost by ignoring heterogeneity.
    ax.scatter(mu_fixed, y_pos[i], s=30, color='C2', marker='|', zorder=6, alpha=0.8, linewidths=2)

    # True effect — fat black tick for reference.
    ax.plot([true_theta_k[k]]*2, [y_pos[i]-0.42, y_pos[i]+0.42],
            color='black', lw=2.5, zorder=7)

    ax.text(-18, y_pos[i], f'n={n_patients[k]}', va='center', ha='right', fontsize=7, color='dimgray')

ax.axvline(0,       color='gray',    ls=':',  lw=1,   alpha=0.4)
ax.axvline(mu_pop,  color='C0',      ls='--', lw=1.5, alpha=0.5)
ax.axvline(true_mu, color='darkred', ls='--', lw=1.5, alpha=0.4)

ax.set_yticks(y_pos)
ax.set_yticklabels([f'T{order[i]+1}' for i in range(K)], fontsize=9)
ax.set_xlabel('Estimated drug effect (mmHg)')
ax.set_title('Forest plot (smallest trials at top)\n'
             'Orange=no-pool  Blue=hierarchical  Green dot=complete pooling  │=truth')
ax.legend(handles=[
    Line2D([0],[0], color='C1', lw=3,     label='No-pooling 95% CI'),
    Line2D([0],[0], color='C0', lw=3,     label='Hierarchical 95% CI'),
    Line2D([0],[0], color='C2', lw=2, marker='|', label='Complete pooling'),
    Line2D([0],[0], color='black', lw=2.5,label='True effect'),
], fontsize=8, loc='lower right')
ax.grid(axis='x', alpha=0.2)

# ── Right panel: simulation — expected error vs trial size ────────────────────
# Run 300 simulated datasets. For each, compute the analytic posterior mean
# (no MCMC needed — fast) and record the per-trial error.
# Using the analytic formula with fitted mu_pop and tau_est as population parameters.
n_sim  = 300
sim_rng = np.random.default_rng(0)  # separate seed from the main data generation

# Reliability (fixed across simulations — depends only on sigma_k, tau_est)
reliability_sim = tau_est**2 / (tau_est**2 + sigma_k**2)

sim_err_nop  = np.zeros((n_sim, K))
sim_err_hier = np.zeros((n_sim, K))

for s in range(n_sim):
    # Draw new true effects from the population distribution
    sim_theta = sim_rng.normal(true_mu, true_tau, size=K)
    # Draw new noisy observations with the same trial sizes
    sim_y     = sim_rng.normal(sim_theta, sigma_k)

    # No-pooling estimate: just the raw observation (flat prior → posterior = data)
    sim_nop  = sim_y

    # Hierarchical estimate: analytic shrinkage formula using fitted population params.
    # θ̂_k = μ_pop + reliability × (y_k − μ_pop)
    sim_hier = mu_pop + reliability_sim * (sim_y - mu_pop)

    sim_err_nop[s]  = np.abs(sim_nop  - sim_theta)
    sim_err_hier[s] = np.abs(sim_hier - sim_theta)

# Average error across all 300 runs — this removes single-run noise.
mean_err_nop  = sim_err_nop.mean(axis=0)
mean_err_hier = sim_err_hier.mean(axis=0)

ax = axes[1]
for k in range(K):
    # Green line = hierarchical improved; red = rare case where it did not.
    improved = mean_err_hier[k] < mean_err_nop[k]
    ax.plot([n_patients[k]]*2, [mean_err_nop[k], mean_err_hier[k]],
            color='limegreen' if improved else 'tomato', lw=2, alpha=0.7, zorder=2)

ax.scatter(n_patients, mean_err_nop,  s=80, color='C1', zorder=4,
           label='No-pooling (expected)',  alpha=0.9)
ax.scatter(n_patients, mean_err_hier, s=80, color='C0', zorder=5,
           label='Hierarchical (expected)', alpha=0.9, marker='D')

ax.set_xscale('log')
ax.set_xlabel('Trial size n  (log scale)')
ax.set_ylabel('Mean |error|  (mmHg)\naveraged over 300 simulated datasets')
ax.set_title(f'Expected per-trial error ({n_sim} simulated datasets)\n'
             f'Hierarchical ◆ is lower than no-pooling ● for every trial size')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.suptitle('Partial pooling: shrinkage is large for small trials, negligible for large ones',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# ── Numerical summary ─────────────────────────────────────────────────────────
print(f"Expected error reduction from partial pooling ({n_sim} simulations):")
print(f"  {'n':>5}  {'No-pool MAE':>12}  {'Hier MAE':>10}  {'Improvement':>12}")
print("  " + "-" * 45)
for k in np.argsort(n_patients):
    impr = (mean_err_nop[k] - mean_err_hier[k]) / mean_err_nop[k] * 100
    print(f"  {n_patients[k]:5d}  {mean_err_nop[k]:12.2f}  {mean_err_hier[k]:10.2f}  {impr:+10.1f}%")


## Key takeaways

1. **Small trials are noisy, not wrong.** A trial with n=12 isn't useless — the hierarchical model uses it, but discounts it appropriately by pulling its estimate toward the population mean.

2. **The population parameters are inferred, not assumed.** The model learns $\mu$ (overall effect) and $\tau$ (heterogeneity) from the ensemble of trials. More trials → better estimates of both.

3. **Shrinkage is optimal under squared error loss.** The Stein paradox says that for three or more parameters estimated jointly, shrinkage toward a shared mean always reduces total error — even when the parameters are independent. Partial pooling is the Bayesian implementation of this idea.

4. **$\tau$ encodes real biology.** If $\tau \approx 0$, trials are all measuring the same thing (complete pooling is fine). If $\tau$ is large, trials are genuinely different and pooling less. In real meta-analyses, a large estimated $\tau$ is a scientific finding — it means the drug effect is heterogeneous and you should study why.

5. **This is a latent variable model.** Each trial's true effect $\theta_k$ is a hidden variable inferred from the data — exactly the framework from the lecture, and exactly what the VAE does in Session 2b.

---

## Exercises

**Q1:** Increase `true_tau` from 4 to 15 (high heterogeneity — trials are very different).  
What happens to shrinkage? Does the hierarchical model still outperform no-pooling for small trials?

**Q2:** Set all `n_patients` to the same value (e.g. `np.full(K, 50)`).  
Does shrinkage still happen? Is it equal for all trials?

**Q3:** In a real meta-analysis there is also **publication bias** — small trials with null results are less likely to be published. What does it do to the population mean estimate?

---

## Answer 1 — High heterogeneity (true_tau = 15 mmHg)

When trials are genuinely very different from each other ($\tau = 15$ mmHg),  
the between-trial heterogeneity is large relative to most individual trial standard errors.

**What the shrinkage formula predicts:**

With $\tau = 15$ and a small trial with $\sigma_k \approx 5$ mmHg:

$$B_k = \frac{\sigma_k^2}{\sigma_k^2 + \tau^2} = \frac{25}{25 + 225} \approx 10\%$$

Even the smallest trials are only shrunk by ~10% — the model correctly infers that  
the observed spread across trials is mostly *real biology*, not measurement noise.

**Key insight:** The advantage of partial pooling over no-pooling shrinks when $\tau$ is large,  
because each trial truly is different and the model is (correctly) trusting each trial's own data more.  
The model adapts automatically — it learns $\tau$ from the data, and applies less shrinkage when $\tau$ is large.

In [ ]:
import logging, pymc as pm, arviz as az, numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
logging.getLogger('pymc').setLevel(logging.ERROR)
logging.getLogger('pytensor').setLevel(logging.ERROR)

# ── Regenerate data with higher between-trial heterogeneity ───────────────────
rng1 = np.random.default_rng(42)
K1           = 12
true_mu1     = 10.0
true_tau1    = 15.0          # ← HIGH heterogeneity: trials genuinely very different
sigma_patient = 20.0
n_patients1  = np.array([15, 400, 25, 300, 12, 500, 50, 80, 120, 20, 200, 30])
true_theta1  = rng1.normal(true_mu1, true_tau1, size=K1)  # now spread over ±15 mmHg
sigma_k1     = sigma_patient / np.sqrt(n_patients1)
y_obs1       = rng1.normal(true_theta1, sigma_k1)

# ── Fit no-pooling model ───────────────────────────────────────────────────────
with pm.Model():
    theta = pm.Normal('theta_k', mu=0, sigma=100, shape=K1)
    pm.Normal('y', mu=theta, sigma=sigma_k1, observed=y_obs1)
    idata_np1 = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                           progressbar=False, target_accept=0.9)

# ── Fit hierarchical model (wider tau prior to match the new scenario) ─────────
with pm.Model():
    mu  = pm.Normal('mu',  mu=0, sigma=20)
    tau = pm.HalfNormal('tau', sigma=20)  # wider prior since we expect more heterogeneity
    theta = pm.Normal('theta_k', mu=mu, sigma=tau, shape=K1)
    pm.Normal('y', mu=theta, sigma=sigma_k1, observed=y_obs1)
    idata_h1 = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                          progressbar=False, target_accept=0.9)

# ── Extract posterior means ────────────────────────────────────────────────────
mu_np1  = idata_np1.posterior['theta_k'].mean(dim=('chain','draw')).values
mu_h1   = idata_h1.posterior['theta_k'].mean(dim=('chain','draw')).values
mu_pop1 = float(idata_h1.posterior['mu'].mean())
tau1    = float(idata_h1.posterior['tau'].mean())

# Shrinkage fraction B_k = σ_k² / (σ_k² + τ²) — should be small when τ is large
B1 = sigma_k1**2 / (sigma_k1**2 + tau1**2)

print(f"Estimated τ = {tau1:.1f} mmHg  (true = {true_tau1})")
print(f"Shrinkage fractions B_k: min = {B1.min()*100:.0f}%  max = {B1.max()*100:.0f}%")
print(f"Compare to τ=4 case: shrinkage was 5%–68%. Now it is much smaller.")
print()
print(f"Mean |error| — no-pool: {np.abs(mu_np1-true_theta1).mean():.2f}  |  "
      f"hier: {np.abs(mu_h1-true_theta1).mean():.2f}")

# ── Forest + error plot ────────────────────────────────────────────────────────
order1   = np.argsort(n_patients1)
hdi_np1  = az.hdi(idata_np1, var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values
hdi_h1   = az.hdi(idata_h1,  var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values
y_pos    = np.arange(K1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
for i, k in enumerate(order1):
    ax.plot([hdi_np1[k,0], hdi_np1[k,1]], [y_pos[i]+0.2]*2, color='C1', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_np1[k], y_pos[i]+0.2, s=55, color='C1', zorder=5)
    ax.plot([hdi_h1[k,0],  hdi_h1[k,1]],  [y_pos[i]-0.2]*2, color='C0', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_h1[k],  y_pos[i]-0.2, s=55, color='C0', zorder=5)
    ax.plot([true_theta1[k]]*2, [y_pos[i]-0.38, y_pos[i]+0.38], color='black', lw=2.5, zorder=6)
ax.axvline(mu_pop1, color='C0', ls='--', lw=1.5, alpha=0.5)
ax.axvline(true_mu1, color='darkred', ls='--', lw=1.5, alpha=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'T{order1[i]+1} (n={n_patients1[order1[i]]})'  for i in range(K1)], fontsize=8)
ax.set_xlabel('Estimated drug effect (mmHg)')
ax.set_title(f'Forest plot — high heterogeneity (τ_true={true_tau1})\nNote wide spread of true effects (black ticks)')
ax.legend(handles=[Line2D([0],[0],color='C1',lw=3,label='No-pooling'),
                   Line2D([0],[0],color='C0',lw=3,label='Hierarchical'),
                   Line2D([0],[0],color='black',lw=2.5,label='True')], fontsize=8)
ax.grid(axis='x', alpha=0.2)

# Error plot: |estimate - truth| vs trial size
err_np1 = np.abs(mu_np1 - true_theta1)
err_h1  = np.abs(mu_h1  - true_theta1)
ax = axes[1]
for k in range(K1):
    ax.plot([n_patients1[k]]*2, [err_np1[k], err_h1[k]],
            color='limegreen' if err_h1[k] < err_np1[k] else 'tomato', lw=1.5, alpha=0.6, zorder=2)
ax.scatter(n_patients1, err_np1, s=70, color='C1', label='No-pooling',   alpha=0.9)
ax.scatter(n_patients1, err_h1,  s=70, color='C0', label='Hierarchical', alpha=0.9, marker='D')
ax.set_xscale('log')
ax.set_xlabel('Trial size n (log scale)')
ax.set_ylabel('|Estimate − true effect| (mmHg)')
ax.set_title(f'Error per trial (τ_true={true_tau1})\nSmaller gap = less benefit from pooling')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.suptitle(f'High heterogeneity (τ={true_tau1} mmHg): less shrinkage, smaller advantage from pooling',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


---

## Answer 2 — Equal trial sizes (n = 50 for all trials)

When all trials have the same size, $\sigma_k$ is identical for every trial.  
The shrinkage formula $B_k = \sigma_k^2 / (\sigma_k^2 + \tau^2)$ then gives  
the **same shrinkage fraction for every trial**.

**Does shrinkage still happen?** Yes — always, unless $\sigma_k = 0$ (infinitely large trial)  
or $\tau = 0$ (no between-trial variation). Shrinkage depends on the *ratio* $\sigma_k^2 / \tau^2$,  
not on *differences* in trial size.

With $n = 50$: $\sigma_k = 20/\sqrt{50} \approx 2.83$ mmHg.  
With $\tau \approx 4$ mmHg: $B_k = 2.83^2 / (2.83^2 + 4^2) \approx 33\%$.

All trials are pulled 33% toward the population mean. The forest plot CIs will all shrink equally — no longer  
a gradient from small to large trials.

In [ ]:
import logging, pymc as pm, arviz as az, numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
logging.getLogger('pymc').setLevel(logging.ERROR)
logging.getLogger('pytensor').setLevel(logging.ERROR)

# ── Regenerate data with equal trial sizes ────────────────────────────────────
rng2 = np.random.default_rng(42)
K2           = 12
true_mu2     = 10.0
true_tau2    =  4.0
sigma_patient = 20.0
n_patients2  = np.full(K2, 50)          # ← ALL EQUAL: every trial has 50 patients
true_theta2  = rng2.normal(true_mu2, true_tau2, size=K2)
sigma_k2     = sigma_patient / np.sqrt(n_patients2)   # identical for all trials
y_obs2       = rng2.normal(true_theta2, sigma_k2)

# ── Fit both models ────────────────────────────────────────────────────────────
with pm.Model():
    theta = pm.Normal('theta_k', mu=0, sigma=100, shape=K2)
    pm.Normal('y', mu=theta, sigma=sigma_k2, observed=y_obs2)
    idata_np2 = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                           progressbar=False, target_accept=0.9)

with pm.Model():
    mu  = pm.Normal('mu',  mu=0, sigma=20)
    tau = pm.HalfNormal('tau', sigma=10)
    theta = pm.Normal('theta_k', mu=mu, sigma=tau, shape=K2)
    pm.Normal('y', mu=theta, sigma=sigma_k2, observed=y_obs2)
    idata_h2 = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                          progressbar=False, target_accept=0.9)

mu_np2  = idata_np2.posterior['theta_k'].mean(dim=('chain','draw')).values
mu_h2   = idata_h2.posterior['theta_k'].mean(dim=('chain','draw')).values
mu_pop2 = float(idata_h2.posterior['mu'].mean())
tau2    = float(idata_h2.posterior['tau'].mean())

# Shrinkage fraction — should be the same for every trial since sigma_k is constant
B2 = sigma_k2**2 / (sigma_k2**2 + tau2**2)
print(f"All trials: n=50, σ_k = {sigma_k2[0]:.2f} mmHg (identical for all)")
print(f"Estimated τ = {tau2:.2f} mmHg")
print(f"Shrinkage fraction: {B2[0]*100:.0f}% — same for every trial (as expected)")
print(f"  (For comparison: with original mixed n, shrinkage ranged from ~5% to ~68%)")

# ── Forest + error plot ────────────────────────────────────────────────────────
order2   = np.arange(K2)   # trials already "sorted" — no size difference
hdi_np2  = az.hdi(idata_np2, var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values
hdi_h2   = az.hdi(idata_h2,  var_names=['theta_k'], hdi_prob=0.95)['theta_k'].values
y_pos    = np.arange(K2)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
for i, k in enumerate(order2):
    ax.plot([hdi_np2[k,0], hdi_np2[k,1]], [y_pos[i]+0.2]*2, color='C1', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_np2[k], y_pos[i]+0.2, s=55, color='C1', zorder=5)
    ax.plot([hdi_h2[k,0],  hdi_h2[k,1]],  [y_pos[i]-0.2]*2, color='C0', lw=3, alpha=0.7, solid_capstyle='round')
    ax.scatter(mu_h2[k],  y_pos[i]-0.2, s=55, color='C0', zorder=5)
    ax.plot([true_theta2[k]]*2, [y_pos[i]-0.38, y_pos[i]+0.38], color='black', lw=2.5, zorder=6)
ax.axvline(mu_pop2, color='C0', ls='--', lw=1.5, alpha=0.5)
ax.axvline(true_mu2, color='darkred', ls='--', lw=1.5, alpha=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'T{k+1}' for k in order2], fontsize=9)
ax.set_xlabel('Estimated drug effect (mmHg)')
ax.set_title(f'Forest plot — all n=50\nShrinkage fraction ≈ {B2[0]*100:.0f}% for every trial')
ax.legend(handles=[Line2D([0],[0],color='C1',lw=3,label='No-pooling'),
                   Line2D([0],[0],color='C0',lw=3,label='Hierarchical'),
                   Line2D([0],[0],color='black',lw=2.5,label='True')], fontsize=8)
ax.grid(axis='x', alpha=0.2)

err_np2 = np.abs(mu_np2 - true_theta2)
err_h2  = np.abs(mu_h2  - true_theta2)
ax = axes[1]
# Jitter x-positions slightly so overlapping points are visible
jitter = (np.arange(K2) - K2//2) * 1.5
for k in range(K2):
    ax.plot([n_patients2[k] + jitter[k]]*2, [err_np2[k], err_h2[k]],
            color='limegreen' if err_h2[k] < err_np2[k] else 'tomato', lw=1.5, alpha=0.6, zorder=2)
ax.scatter(n_patients2 + jitter, err_np2, s=70, color='C1', label='No-pooling',   alpha=0.9)
ax.scatter(n_patients2 + jitter, err_h2,  s=70, color='C0', label='Hierarchical', alpha=0.9, marker='D')
ax.set_xlabel('Trial (all n=50; jittered for visibility)')
ax.set_ylabel('|Estimate − true effect| (mmHg)')
ax.set_title('Error per trial — shrinkage is uniform for all trials')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.suptitle('Equal trial sizes: CIs shrink uniformly — no size-dependent gradient',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f"Mean |error|: no-pool = {err_np2.mean():.2f} mmHg  |  hier = {err_h2.mean():.2f} mmHg")


---

## Answer 3 — Publication bias

### What publication bias does to the estimate

The hierarchical model is fit to the **observed** studies only. If negative small studies are absent,  
the model sees a biased sample. The inferred population mean $\mu$ is pulled upward — not because  
the drug is more effective, but because negative evidence is missing.

**This is a fundamental limitation of all meta-analytic approaches**, hierarchical or not:  
they pool the information they are given. Garbage in, garbage out.

### What to do about it

- **Egger's test**: a regression of effect size on precision — significant asymmetry detects bias
- **Selection models**: explicitly model the probability that a study is published as a function of its result
- **Pre-registration**: the only approach that *prevents* the bias rather than correcting for it —  
  all trials must register their outcomes before they start, so null results cannot quietly disappear

This is an active area of research in meta-science, and a strong motivation for clinical trial registries  
(ClinicalTrials.gov, ISRCTN) — which were partly inspired by the very beta-blocker meta-analyses  
that motivated this notebook.